In [1]:
import os
import re
import pandas as pd

def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    # Eliminar saludos/despedidas comunes
    texto = re.sub(r'(estimado|estimada|señores|atentamente|saludos).*', '', texto)
    # Reemplazar RUTs o números largos por un token genérico (anonimización)
    texto = re.sub(r'\b\d{7,k}\b', '[NUM]', texto) 
    # Quitar caracteres especiales excesivos, dejando puntuación básica
    texto = re.sub(r'[^a-záéíóúñ0-9\s.,;]', ' ', texto)
    # Normalizar espacios
    return re.sub(r'\s+', ' ', texto).strip()

# ------- PARAMETROS y ARCHIVOS -------
RUTA_CATEGORIAS = "categorias.txt"
RUTA_RECLAMOS = "textos_reclamos_resp_empresa.xlsx"
COLS = [
    "Tema - Peticion Concreta",
    "Texto",
    "Repuesta de Empresa (1era)",
    "Clasificacion que le asignó el Analista "
]

# 1. Cargar y limpiar categorías
with open(RUTA_CATEGORIAS, 'r', encoding='utf-8') as f:
    CATEGORIAS = [line.strip() for line in f if line.strip()]

df_reclamos = pd.read_excel(RUTA_RECLAMOS, skiprows=1, usecols=COLS).dropna()

df_reclamos['peticion_limpia'] = df_reclamos['Tema - Peticion Concreta'].apply(limpiar_texto)
df_reclamos['texto_limpio'] = df_reclamos['Texto'].apply(limpiar_texto)
df_reclamos['respuesta_limpia'] = df_reclamos['Repuesta de Empresa (1era)'].apply(limpiar_texto)

df_reclamos['texto_modelo'] = (
    df_reclamos['texto_limpio']  + " [SEP] " + 
    df_reclamos['peticion_limpia']+ " [SEP] " + 
    df_reclamos['respuesta_limpia']
)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ==============================================================================
# 2. Preparación de etiquetas y datos (Con filtro de clases minoritarias)
# ==============================================================================

# A. Identificar y filtrar clases con menos de 2 ejemplos
columna_clasificacion = 'Clasificacion que le asignó el Analista '
conteo_clases = df_reclamos[columna_clasificacion].value_counts()

# Nos quedamos solo con las categorías que tienen 2 o más reclamos
clases_validas = conteo_clases[conteo_clases >= 2].index
df_reclamos = df_reclamos[df_reclamos[columna_clasificacion].isin(clases_validas)]

print(f"Total de reclamos tras filtrar categorías con 1 solo caso: {len(df_reclamos)}")

# B. Mapeamos las categorías de texto a números
label_encoder = LabelEncoder()
df_reclamos['label'] = label_encoder.fit_transform(df_reclamos[columna_clasificacion])
num_labels = len(label_encoder.classes_)

# C. División de datos (Entrenamiento y Validación)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_reclamos['texto_modelo'].tolist(), 
    df_reclamos['label'].tolist(), 
    test_size=0.2, 
    random_state=42,
    stratify=df_reclamos['label'].tolist() # Ahora funcionará sin problemas
)

# 3. Configuración del Tokenizer y Modelo (BETO - Spanish BERT)
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReclamosDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        encoding = self.tokenizer(
            self.texts[item],
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

# 4. DataLoaders
train_dataset = ReclamosDataset(train_texts, train_labels, tokenizer)
val_dataset = ReclamosDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

# 5. Inicialización del Modelo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels, use_safetensors=True)
model.to(device)

# 6. Optimizador y Scheduler
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * 3  # 3 épocas
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# 7. Pesos por categoría
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights_tensor)

# 8. Bucle de Entrenamiento
print("Iniciando entrenamiento...")
for epoch in range(8):
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
    
    print(f"Época {epoch + 1} completada. Loss de Entrenamiento: {loss.item():.4f}")

# ==============================================================================
# 9. EVALUACIÓN Y MÉTRICAS DE DESEMPEÑO (Para validación del flujo)
# ==============================================================================
print("\nIniciando evaluación del modelo con el conjunto de validación...")
model.eval()

todas_predicciones = []
todos_valores_reales = []

# Iterar sobre validación sin calcular gradientes
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask)
        
        # Obtener la clase con mayor probabilidad
        predicciones = torch.argmax(outputs.logits, dim=-1)
        
        todas_predicciones.extend(predicciones.cpu().numpy())
        todos_valores_reales.extend(labels.cpu().numpy())

nombres_clases = label_encoder.classes_

# --- Generación de Reporte Textual ---
print("\n" + "="*55)
print(" REPORTE DE CLASIFICACIÓN - ASISTENTE SEC")
print("="*55)

accuracy = accuracy_score(todos_valores_reales, todas_predicciones)
print(f"\nExactitud Global (Accuracy): {accuracy * 100:.2f}%\n")

reporte = classification_report(
    todos_valores_reales, 
    todas_predicciones, 
    target_names=nombres_clases,
    zero_division=0
)
print(reporte)

# --- Generación de Matriz de Confusión Visual ---
matriz_conf = confusion_matrix(todos_valores_reales, todas_predicciones)

plt.figure(figsize=(10, 8))
sns.heatmap(
    matriz_conf, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=nombres_clases, 
    yticklabels=nombres_clases
)

plt.title('Matriz de Confusión - Validación de Reclamos SEC', fontsize=14)
plt.ylabel('Clasificación Real (Analista Humano)', fontsize=12)
plt.xlabel('Clasificación Predicha (Asistente IA)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Guardar y mostrar
#plt.savefig('matriz_confusion_sec.png', dpi=300)
plt.show()

# 10. Guardar el modelo para el prototipo funcional
# model.save_pretrained("./modelo_clasificador_sec")
# tokenizer.save_pretrained("./modelo_clasificador_sec")